# LoCED 100M body — 12 layers, dynamic mHC + MagicNorm, GPT-2 vocab

This notebook builds and trains a **12-layer LoCED** model with:

- **dynamic mHC** (`mhc_streams=4`, `mhc_dynamic=True`): four residual
  streams, a Sinkhorn-constrained `H_res`, softmax `H_pre` / `H_post`, and
  per-token coefficients predicted from the mean stream;
- **MagicNorm** (`magic_norm=True`): a module-level RMSNorm at every visit
  exit;
- **MoE**: 62 routed experts + 1 shared expert, top-1 routing, `I=168`;
- **GPT-2 vocabulary**: `vocab_size=50,257`.

### Parameter budget

| Piece | Params |
|---|---:|
| 12-layer transformer body | **99.75M** |
| GPT-2 embedding | 12.87M |
| GPT-2 output head | 12.87M |
| **Full model** | **125.48M** |
| **Active body per token** | **5.30M** |

Per your choice, the 100M target is the **transformer body**; the GPT-2
embedding and head sit on top. “Active body” counts attention, norms, mHC,
the router, the shared expert, and the top-1 routed expert for one token,
across all 12 physical layers. It excludes the embedding lookup and the
vocabulary head; including the full head gives 18.17M active. The body is
5.32% active.

### Schedule

- encode / pass A: `12 × full`
- decode / pass B: `6 × reuse + 6 × reindex`
- 24 layer visits per generated token (12 physical layers, weight-tied
  across passes).

### What this notebook is, and is not

- It is a **synthetic-data plumbing and training demo** for Kaggle TPU.
- The reference repo evaluates MoE densely; this notebook installs a local
  top-1 `ragged_dot` dispatch so a token really reads only its selected
  expert. The core repo is not modified.
- It is not a quality or throughput claim. The synthetic task only proves
  that the architecture, parameter budget, gradients, and decode path work.

## 1. Environment

This notebook uses the **pure-JAX reference path**, not MaxText. That keeps
the setup small and avoids touching the TPU JAX installation.

- On Kaggle, select **Accelerator → TPU v5e-8** and run all cells.
- If `loced` is not importable, the cell clones the public repo into
  `/kaggle/working/loced`.
- `optax` is installed only if it is missing. `transformers` is used only to
  verify the GPT-2 tokenizer; if it is unavailable, the notebook falls back to
  `vocab_size=50,257`.

For a quick CPU smoke test, set these environment variables before running:

```text
LOCED_SEQ_LEN=16  LOCED_STEPS=1  LOCED_BATCH=1  LOCED_RUN_GENERATION=0
```

In [ ]:
import functools
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
REPO_URL = os.environ.get("LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git")
REPO_REF = os.environ.get("LOCED_REPO_REF", "master")
REPO_DIR = WORK / "loced"


def _find_loced_root():
    for root in (Path.cwd(), WORK, REPO_DIR):
        if (root / "loced" / "model.py").is_file():
            return root
        if (root / "model.py").is_file() and (root / "block.py").is_file():
            return root.parent
    for path in Path("/kaggle/input").glob("**/loced/model.py"):
        return path.parents[1]
    return None


root = _find_loced_root()
if root is None:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"cloning {REPO_URL} ({REPO_REF}) -> {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    root = REPO_DIR.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

try:
    import optax
except ImportError:
    print("installing optax ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optax"], check=True)
    import optax

try:
    from transformers import GPT2TokenizerFast

    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    VOCAB_SIZE = len(tokenizer)
except Exception as exc:
    tokenizer = None
    VOCAB_SIZE = 50257
    print(f"GPT-2 tokenizer unavailable ({exc}); using vocab_size={VOCAB_SIZE}")

import jax
import jax.numpy as jnp

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("backend :", jax.default_backend())
print("devices :", jax.devices())
print("vocab   :", VOCAB_SIZE)
print("loced   :", root)

## 2. Architecture configuration

The model is deliberately narrow (`d=256`) and deep-ish (`12` physical
layers) so the body fits a 100M budget while staying sparsely activated.

- 12 physical layers, each with attention + MoE + pre-norms + MagicNorm.
- 24 visits: pass A is `12 × full`; pass B is `6 × reuse + 6 × reindex`.
- 4 mHC streams, dynamic coefficients, 10 Sinkhorn iterations.
- 62 routed experts, top-1, one always-on shared expert, `I=168`.
- GPT-2 vocabulary, so the embedding and head are 12.87M each.

In [ ]:
from loced.config import LoCEDConfig

N_LAYERS = 12
D_MODEL = 256
N_HEADS = 8
HEAD_DIM = 32
KV_DIM = 32
SEQ_LEN = int(os.environ.get("LOCED_SEQ_LEN", "128"))
DECODE_LEN = int(os.environ.get("LOCED_DECODE_LEN", "32"))
N_EXPERTS = 62
TOP_K_EXPERTS = 1
INTERMEDIATE = 168

assert SEQ_LEN % 2 == 0, "pool_m=2 requires an even sequence length"
assert DECODE_LEN <= SEQ_LEN

ENCODE_LAYERS = tuple("full" for _ in range(N_LAYERS))
DECODE_LAYERS = (
    ENCODE_LAYERS
    + tuple("reuse" for _ in range(6))
    + tuple("reindex" for _ in range(6))
)

cfg = LoCEDConfig(
    vocab_size=VOCAB_SIZE,
    hidden_size=D_MODEL,
    max_seq_len=SEQ_LEN,
    max_decode_len=DECODE_LEN,
    magic_norm=True,
    mhc_streams=4,
    mhc_sinkhorn_iters=10,
    mhc_dynamic=True,
    num_attention_heads=N_HEADS,
    head_dim=HEAD_DIM,
    kv_dim=KV_DIM,
    pool_m=2,
    window=min(32, SEQ_LEN // 2),
    top_k=min(32, SEQ_LEN // 2),
    indexer_heads=4,
    indexer_dim=64,
    indexer_block=4,
    hier_blocks=4,
    encode_layers=ENCODE_LAYERS,
    decode_layers=DECODE_LAYERS,
    num_experts=N_EXPERTS,
    num_shared_experts=1,
    top_k_experts=TOP_K_EXPERTS,
    intermediate_size=INTERMEDIATE,
)

print("layers      :", cfg.num_layers)
print("visits      :", cfg.num_visits,
      f"(encode {cfg.prefill_visits} + decode {cfg.num_visits - cfg.prefill_visits})")
print("sparse pass :", cfg.sparse_layers)
print("mHC         : streams", cfg.mhc_streams, "dynamic", cfg.mhc_dynamic,
      "sinkhorn", cfg.mhc_sinkhorn_iters)
print("magic norm  :", cfg.magic_norm)
print("MoE         :", cfg.num_experts, "routed, top", cfg.top_k_experts,
      "shared", cfg.num_shared_experts, "I", cfg.intermediate_size)

## 3. Parameter audit

`body` is everything except the GPT-2 token embedding and output head.
`active_body` is the set of body weights one token actually touches:

- attention, norms, mHC, and the router in every physical layer;
- the shared expert in every layer;
- exactly `top_k_experts` routed experts per layer.

The check also projects each visit’s `H_res` through Sinkhorn and verifies
that it is doubly stochastic.

In [ ]:
from loced.block import sinkhorn
from loced.model import count_params, init_model

params = init_model(cfg, jax.random.PRNGKey(0))
total = count_params(params)
embed = int(params["embed"].size)
head = int(params["head"].size)
body = total - embed - head


def count_active_body(cfg, params):
    """Body parameters touched by one token's forward pass."""
    n = 0
    for name in ("W_bk", "W_bv", "W_ik", "W_iq"):
        n += int(params[name].size)
    for name in ("cond", "gate_a", "gate_m", "norm_f",
                 "mhc_res", "mhc_pre", "mhc_post", "mhc_dyn"):
        n += int(params[name].size)
    for cell in params["cells"]:
        for name in ("norm_a", "norm_m", "norm_magic",
                     "W_q", "W_fx", "W_fv", "W_o", "W_gate",
                     "Ws1", "Ws3", "Ws2"):
            n += int(cell[name].size)
        per_expert = int(
            cell["We1"][0].size + cell["We3"][0].size + cell["We2"][0].size
        )
        n += cfg.top_k_experts * per_expert
    return int(n)


active_body = count_active_body(cfg, params)
active_with_head = active_body + head + D_MODEL  # one embedding row + full head
mhc = np.asarray(sinkhorn(params["mhc_res"], cfg.mhc_sinkhorn_iters))
row_err = float(np.max(np.abs(mhc.sum(axis=-1) - 1.0)))
col_err = float(np.max(np.abs(mhc.sum(axis=-2) - 1.0)))

print(f"total parameters          : {total:>14,}  ({total / 1e6:8.3f} M)")
print(f"  transformer body        : {body:>14,}  ({body / 1e6:8.3f} M)")
print(f"  GPT-2 embedding         : {embed:>14,}  ({embed / 1e6:8.3f} M)")
print(f"  GPT-2 output head       : {head:>14,}  ({head / 1e6:8.3f} M)")
print(f"active body per token     : {active_body:>14,}  ({active_body / 1e6:8.3f} M)")
print(f"active incl. head + row   : {active_with_head:>14,}  ({active_with_head / 1e6:8.3f} M)")
print(f"body sparsity             : {active_body / body:.2%} active")
print(f"Sinkhorn max row/col err  : {max(row_err, col_err):.3e}")

assert 95e6 <= body <= 105e6, body
assert 5e6 <= active_body <= 6e6, active_body
assert max(row_err, col_err) < 1e-4

## 4. Sparse top-1 MoE dispatch

The reference `loced.fnn.moe_forward` evaluates **all** experts and masks the
result. That is fine for parity tests, but it makes a top-1 MoE pay for all 62
experts.

For this notebook we install a local top-1 dispatch built on
`jax.lax.ragged_dot`: tokens are sorted by their selected expert, each expert
runs only on its own tokens, and the outputs are scattered back. For top-1
this is mathematically identical to the dense kernel. A tiny dense-vs-sparse
parity check runs before the patch is installed.

The core repository is unchanged; this override lives only in the notebook.

In [ ]:
import dataclasses

import loced.block as loced_block
from loced.fnn import _swiglu, init_moe
from loced.train import tiny_config

DENSE_MOE_FORWARD = loced_block.moe_forward

if cfg.top_k_experts != 1:
    raise ValueError("this notebook's sparse dispatch assumes top_k_experts == 1")


def sparse_top1_moe(cfg, p, x):
    """Top-1 MoE: each token reads only its selected expert's weights."""
    B, T, d = x.shape
    shared = _swiglu(x @ p["Ws1"], x @ p["Ws3"]) @ p["Ws2"]

    xf = x.reshape(B * T, d)
    probs = jax.nn.softmax(xf @ p["W_gate"], axis=-1)
    top_w, top_idx = jax.lax.top_k(probs, 1)
    top_w = top_w / jnp.maximum(top_w.sum(-1, keepdims=True), 1e-9)
    flat_idx = top_idx[:, 0]

    perm = jnp.argsort(flat_idx)
    inv = jnp.argsort(perm)
    xs = xf[perm]
    sizes = jnp.bincount(flat_idx, length=cfg.num_experts)

    a = jax.lax.ragged_dot(xs, p["We1"], sizes)
    g = jax.lax.ragged_dot(xs, p["We3"], sizes)
    h = _swiglu(a, g)
    y = jax.lax.ragged_dot(h, p["We2"], sizes)
    routed = (y * top_w[:, 0][:, None])[inv].reshape(B, T, d)

    frac = jnp.mean(
        jax.nn.one_hot(flat_idx, cfg.num_experts, dtype=jnp.float32), axis=0
    )
    mean_p = jnp.mean(probs, axis=0)
    aux = cfg.num_experts * jnp.sum(frac * mean_p)
    return shared + routed, aux


# Dense-vs-sparse parity on a tiny MoE layer (values and gradients).
tiny = dataclasses.replace(tiny_config(), top_k_experts=1)
tiny_params = init_moe(tiny, jax.random.PRNGKey(1))
tiny_x = jax.random.normal(jax.random.PRNGKey(2), (2, 8, tiny.hidden_size))

dense_out, dense_aux = DENSE_MOE_FORWARD(tiny, tiny_params, tiny_x)
sparse_out, sparse_aux = sparse_top1_moe(tiny, tiny_params, tiny_x)
value_err = float(jnp.max(jnp.abs(dense_out - sparse_out)))
aux_err = float(jnp.abs(dense_aux - sparse_aux))

dense_grad = jax.grad(
    lambda q: DENSE_MOE_FORWARD(tiny, q, tiny_x)[0].sum()
)(tiny_params)
sparse_grad = jax.grad(
    lambda q: sparse_top1_moe(tiny, q, tiny_x)[0].sum()
)(tiny_params)
grad_err = max(
    float(jnp.max(jnp.abs(a - b)))
    for a, b in zip(jax.tree.leaves(dense_grad), jax.tree.leaves(sparse_grad))
)

print(f"top-1 parity: value={value_err:.3e} aux={aux_err:.3e} grad={grad_err:.3e}")
assert value_err < 1e-6 and aux_err < 1e-6 and grad_err < 1e-5

loced_block.moe_forward = sparse_top1_moe
print("patched loced.block.moe_forward -> sparse_top1_moe")

## 5. Synthetic GPT-2-token data

The model has a full 50,257-way GPT-2 head, but the synthetic task only uses
the first `ACTIVE_VOCAB=512` token IDs. That keeps the task learnable while
the embedding/head shapes and the tokenizer contract stay GPT-2-compatible.

The rule is local and causal: with probability `rule_p`, the next token is
`(previous + 1) % ACTIVE_VOCAB`; otherwise it is uniform noise. A model that
learns the rule should reach high next-token accuracy and generate the
successor sequence greedily.

In [ ]:
ACTIVE_VOCAB = min(512, VOCAB_SIZE)


def synthetic_batch_gpt2(rng, batch, seq, rule_p=0.9):
    """Rule-based next-token task over a subset of GPT-2 token IDs."""
    tok = np.zeros((batch, seq), dtype=np.int32)
    tok[:, 0] = rng.integers(0, ACTIVE_VOCAB, size=batch)
    for t in range(1, seq):
        step = (tok[:, t - 1] + 1) % ACTIVE_VOCAB
        noise = rng.integers(0, ACTIVE_VOCAB, size=batch)
        take_rule = rng.random(batch) < rule_p
        tok[:, t] = np.where(take_rule, step, noise)
    return jnp.asarray(tok)


sample_rng = np.random.default_rng(0)
sample = synthetic_batch_gpt2(sample_rng, 2, min(16, SEQ_LEN))
print("sample token ids:", np.asarray(sample)[0].tolist())
if tokenizer is not None:
    print("decoded sample :", tokenizer.decode(np.asarray(sample)[0].tolist()))

## 6. Training on Kaggle TPU

Training uses Muon on matrix weights and AdamW on embeddings, the head,
routers, and 1-D parameters. Gradients are all-reduced across the TPU mesh
with `jax.shard_map`; parameters are replicated and the batch is sharded over
the `data` axis.

Defaults:

- global batch 8 (one sequence per chip on v5e-8);
- 100 steps on TPU, 5 on CPU;
- Muon/AdamW learning rate `1e-3`;
- `reuse_p` stays 0 for the first 20% of steps, ramps to 1 over the next 30%,
  then stays at 1. The early steps train the exact FULL-attention oracle; the
  later steps train the sparse indexer-selection path.

If you hit memory limits, lower `LOCED_BATCH` or `LOCED_SEQ_LEN`.

In [ ]:
from loced.model import forward_train
from loced.train import make_optimizer

N_DEVICES = len(jax.devices())
DEFAULT_BATCH = 8 if N_DEVICES >= 8 else max(1, 2 * N_DEVICES)
GLOBAL_BATCH = int(os.environ.get("LOCED_BATCH", str(DEFAULT_BATCH)))
GLOBAL_BATCH = max(N_DEVICES, (GLOBAL_BATCH // N_DEVICES) * N_DEVICES)
PER_DEVICE_BATCH = GLOBAL_BATCH // N_DEVICES
DEFAULT_STEPS = 100 if jax.default_backend() != "cpu" else 5
STEPS = int(os.environ.get("LOCED_STEPS", str(DEFAULT_STEPS)))
LR = float(os.environ.get("LOCED_LR", "1e-3"))

print(f"devices={N_DEVICES}  global_batch={GLOBAL_BATCH}  "
      f"per_device_batch={PER_DEVICE_BATCH}  steps={STEPS}  lr={LR}")

optimizer = make_optimizer(cfg, params, optim="muon", lr=LR)
opt_state = optimizer.init(params)

from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

mesh = Mesh(np.array(jax.devices()), axis_names=("data",))
replicated = NamedSharding(mesh, P())
batch_sharded = NamedSharding(mesh, P("data"))

params = jax.device_put(params, replicated)
opt_state = jax.device_put(opt_state, replicated)


@jax.jit
@jax.shard_map(
    mesh=mesh,
    in_specs=(P(), P(), P("data"), P()),
    out_specs=(P(), P(), P()),
)
def train_step(params, state, tokens, reuse_p):
    def loss_fn(p):
        out = forward_train(cfg, p, tokens, reuse_p=reuse_p)
        metrics = {k: out[k] for k in ("loss", "ce", "ce_aux", "kl", "moe_aux")}
        metrics["reuse_p"] = jnp.asarray(reuse_p, jnp.float32)
        return out["loss"], metrics

    (loss, metrics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    grads = jax.lax.pmean(grads, "data")
    metrics = jax.lax.pmean(metrics, "data")

    updates, state = optimizer.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    return params, state, metrics


warmup = int(STEPS * cfg.reuse_warmup_frac) if cfg.reuse_onpolicy else STEPS
ramp = max(1, int(STEPS * cfg.reuse_ramp_frac))
history = []
train_rng = np.random.default_rng(1)

for i in range(STEPS):
    reuse_p = (
        0.0
        if not cfg.reuse_onpolicy
        else min(max((i - warmup) / ramp, 0.0), 1.0)
    )
    tokens = synthetic_batch_gpt2(train_rng, GLOBAL_BATCH, SEQ_LEN, rule_p=0.9)
    tokens = jax.device_put(tokens, batch_sharded)
    reuse = jax.device_put(jnp.asarray(reuse_p, jnp.float32), replicated)

    t0 = time.perf_counter()
    params, opt_state, metrics = train_step(params, opt_state, tokens, reuse)
    row = {k: float(v) for k, v in metrics.items()}
    row["ms_step"] = (time.perf_counter() - t0) * 1000.0
    row["step"] = i
    history.append(row)

    if i == 0 or (i + 1) % max(1, STEPS // 10) == 0 or i == STEPS - 1:
        print(
            f"step {i:3d}  loss {row['loss']:.4f}  ce {row['ce']:.4f}  "
            f"kl {row['kl']:.4f}  moe_aux {row['moe_aux']:.4f}  "
            f"reuse_p {row['reuse_p']:.2f}  {row['ms_step']:.1f} ms"
        )

print("training done")

## 7. Evaluation and generation

The trained model is evaluated at `reuse_p = 0.0`, `0.5`, and `1.0`. The
`reuse_p=1.0` path uses the sparse indexer selection that inference uses.

Generation is greedy and optional; it is skipped on CPU by default because
the unrolled decode loop is slow there. Set `LOCED_RUN_GENERATION=1` to force
it.

In [ ]:
params_trained = jax.tree.map(
    lambda x: jax.device_put(x, jax.devices()[0]), params
)


@jax.jit
def eval_forward(p, tokens, reuse_p):
    out = forward_train(cfg, p, tokens, reuse_p=reuse_p)
    pred = jnp.argmax(out["logits"][:, :-1], axis=-1)
    acc = jnp.mean((pred == tokens[:, 1:]).astype(jnp.float32))
    return out["loss"], out["ce"], out["kl"], acc


eval_rng = np.random.default_rng(123)
eval_tokens = synthetic_batch_gpt2(eval_rng, 4, SEQ_LEN, rule_p=0.9)
eval_results = {}
for reuse_p in (0.0, 0.5, 1.0):
    loss, ce, kl, acc = eval_forward(params_trained, eval_tokens, reuse_p)
    eval_results[str(reuse_p)] = {
        "loss": float(loss),
        "ce": float(ce),
        "kl": float(kl),
        "next_token_acc": float(acc),
    }
    print(
        f"reuse_p={reuse_p:.1f}  loss={float(loss):.4f}  "
        f"ce={float(ce):.4f}  kl={float(kl):.4f}  "
        f"next-token acc={float(acc):.3f}"
    )

RUN_GENERATION = os.environ.get(
    "LOCED_RUN_GENERATION", "0" if jax.default_backend() == "cpu" else "1"
) == "1"
generation = None
if RUN_GENERATION:
    from loced.model import generate

    max_new = int(os.environ.get("LOCED_MAX_NEW", "16"))
    prompt = synthetic_batch_gpt2(
        np.random.default_rng(7), 1, min(16, SEQ_LEN // 2), rule_p=0.9
    )
    t0 = time.perf_counter()
    gen = generate(cfg, params_trained, prompt, max_new)
    gen_ids = np.asarray(gen)[0]
    rule_acc = float(np.mean((gen_ids[1:] == (gen_ids[:-1] + 1) % ACTIVE_VOCAB))) if len(gen_ids) > 1 else 0.0
    generation = {
        "prompt_ids": np.asarray(prompt)[0].tolist(),
        "generated_ids": gen_ids.tolist(),
        "greedy_rule_accuracy": rule_acc,
        "seconds": time.perf_counter() - t0,
    }
    if tokenizer is not None:
        generation["generated_text"] = tokenizer.decode(gen_ids.tolist())
    print("prompt    :", generation["prompt_ids"])
    print("generated :", generation["generated_ids"])
    print(f"greedy rule accuracy: {rule_acc:.3f}")
    if tokenizer is not None:
        print("text      :", generation["generated_text"])
else:
    print("generation skipped (set LOCED_RUN_GENERATION=1 to enable)")

## 8. Machine-readable report

The report includes the environment, config, parameter budget, training
history, eval metrics, and optional generation output. It is written to
`/kaggle/working/loced_100m_body_12l_mhc_gpt2_synth_report.json`.

In [ ]:
def _git_commit(path):
    try:
        return subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "--short", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        return None


report = {
    "kind": "loced_100m_body_12l_dynamic_mhc_magicnorm_gpt2_synth",
    "date": "2026-11-05",
    "repo_commit": _git_commit(root),
    "backend": jax.default_backend(),
    "devices": N_DEVICES,
    "jax_version": jax.__version__,
    "python_version": sys.version.split()[0],
    "tokenizer": "gpt2" if tokenizer is not None else None,
    "vocab_size": VOCAB_SIZE,
    "config": {
        "layers": cfg.num_layers,
        "visits": cfg.num_visits,
        "d_model": cfg.hidden_size,
        "heads": cfg.num_attention_heads,
        "head_dim": cfg.head_dim,
        "kv_dim": cfg.kv_dim,
        "seq_len": cfg.max_seq_len,
        "decode_len": cfg.max_decode_len,
        "magic_norm": cfg.magic_norm,
        "mhc_streams": cfg.mhc_streams,
        "mhc_dynamic": cfg.mhc_dynamic,
        "mhc_sinkhorn_iters": cfg.mhc_sinkhorn_iters,
        "num_experts": cfg.num_experts,
        "top_k_experts": cfg.top_k_experts,
        "shared_experts": cfg.num_shared_experts,
        "intermediate_size": cfg.intermediate_size,
        "sparse_layers": list(cfg.sparse_layers),
    },
    "params": {
        "total": int(total),
        "body": int(body),
        "embedding": int(embed),
        "head": int(head),
        "active_body_per_token": int(active_body),
        "active_with_head_and_embed_row": int(active_with_head),
        "body_active_fraction": float(active_body / body),
        "sinkhorn_max_error": float(max(row_err, col_err)),
    },
    "training": {
        "steps": STEPS,
        "global_batch": GLOBAL_BATCH,
        "per_device_batch": PER_DEVICE_BATCH,
        "seq_len": SEQ_LEN,
        "learning_rate": LR,
        "optimizer": "muon",
        "history": history,
    },
    "eval": eval_results,
    "generation": generation,
    "notes": [
        "100M target is the transformer body; GPT-2 embedding and head are on top.",
        "active_body excludes the embedding lookup and vocabulary head.",
        "MoE uses a notebook-local sparse top-1 ragged_dot dispatch.",
        "Synthetic data only; this is not a quality or throughput claim.",
    ],
}

report_path = WORK / "loced_100m_body_12l_mhc_gpt2_synth_report.json"
report_path.write_text(json.dumps(report, indent=2, default=float))
print("wrote", report_path)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot([r["step"] for r in history], [r["loss"] for r in history])
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.set_title("LoCED 100M body — synthetic training")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plot_path = WORK / "loced_100m_body_loss.png"
    fig.savefig(plot_path, dpi=140)
    plt.close(fig)
    print("wrote", plot_path)
except Exception as exc:
    print("plot skipped:", exc)